In [2]:
import pandas as pd
import os

# Load the CSV file
df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores.csv")

df['poster_title'] = df['image_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0].replace('_', ' '))

print(df[['image_path', 'poster_title']].head())

                                          image_path  \
0           poster_images/poster_images/Superman.jpg   
1  poster_images/poster_images/The_Conjuring_Last...   
2              poster_images/poster_images/Ligaw.jpg   
3  poster_images/poster_images/El_Cas_Àngelus_La_...   
4  poster_images/poster_images/Jurassic_World_Reb...   

                           poster_title  
0                              Superman  
1              The Conjuring Last Rites  
2                                 Ligaw  
3  El Cas Àngelus La fascinació de Dalí  
4                Jurassic World Rebirth  


In [3]:
face_matches_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/face_matches.csv")

In [4]:
df[df.image_path == "poster_images/poster_images/_Tis_the_Season_for_Love_2015_.jpg"]

,image_path,imdb_score,score_range,poster_title
29871,poster_images/poster_images/_Tis_the_Season_fo...,6.7,6–7,Tis the Season for Love 2015


In [5]:
df.to_csv("tesnime.csv",index=False)

In [6]:
df.shape

(49143, 4)

In [7]:
image_with_actors_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores_with_actors.csv")

In [8]:
image_with_actors_df = image_with_actors_df[["image_path","popularity"]]


In [9]:
combined_df = pd.merge(
    image_with_actors_df,
    df,
    on='image_path',
    how='right'  # keeps only rows that are in imdb_df
)

# Check result
print(combined_df.head())
print(f"Combined row count: {len(combined_df)}")

                                          image_path  popularity  imdb_score  \
0           poster_images/poster_images/Superman.jpg        15.0         0.0   
1  poster_images/poster_images/The_Conjuring_Last...        15.0         0.0   
2              poster_images/poster_images/Ligaw.jpg        15.0         0.0   
3  poster_images/poster_images/El_Cas_Àngelus_La_...        15.0         0.0   
4  poster_images/poster_images/Jurassic_World_Reb...        15.0         0.0   

  score_range                          poster_title  
0         0–1                              Superman  
1         0–1              The Conjuring Last Rites  
2         0–1                                 Ligaw  
3         0–1  El Cas Àngelus La fascinació de Dalí  
4         0–1                Jurassic World Rebirth  
Combined row count: 49311


In [11]:
bert_embeddings = pd.read_csv("/kaggle/input/embeddings-1/titles_with_bert_embeddings_1.csv")

In [12]:
bert_embeddings['bert_cls_embedding'] = bert_embeddings['bert_cls_embedding'].apply(
    lambda x: [float(i) for i in x.split(',')] if isinstance(x, str) else x
)

# Step 2 (Optional): Check result
print(len(bert_embeddings['bert_cls_embedding'].iloc[0]))# should show a list like [-0.86, -0.12, ...]
print(type(bert_embeddings['bert_cls_embedding'].iloc[0]))  # should be <class 'list'>

768
<class 'list'>


In [13]:
combined_df.shape

(49311, 5)

In [14]:
bert_embeddings= bert_embeddings[["image_path","bert_cls_embedding"]]

In [16]:
final_df = pd.merge(
    combined_df,
    bert_embeddings,
    on='image_path',
    how='right'  # keep only those in bert_embeddings
)

# Step 2: Optional - verify
print(final_df.head())
print(f"Final row count: {len(final_df)}")

                                          image_path  popularity  imdb_score  \
0           poster_images/poster_images/Superman.jpg        15.0         0.0   
1  poster_images/poster_images/The_Conjuring_Last...        15.0         0.0   
2              poster_images/poster_images/Ligaw.jpg        15.0         0.0   
3  poster_images/poster_images/El_Cas_Àngelus_La_...        15.0         0.0   
4  poster_images/poster_images/Jurassic_World_Reb...        15.0         0.0   

  score_range                          poster_title  \
0         0–1                              Superman   
1         0–1              The Conjuring Last Rites   
2         0–1                                 Ligaw   
3         0–1  El Cas Àngelus La fascinació de Dalí   
4         0–1                Jurassic World Rebirth   

                                  bert_cls_embedding  
0  [-0.86040306, -0.120517045, -0.83552206, 0.296...  
1  [-1.0534014, -0.4176956, -0.37494773, -0.19581...  
2  [-0.4294514, -0.1105

In [17]:
final_df.rename(columns={'popularity': 'actor_score'}, inplace=True)
final_df.rename(columns={'bert_cls_embedding': 'title_embedding'}, inplace=True)

In [18]:
!pip install lightning --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 15.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 86.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 re

In [19]:
# ────────────────────────────────────────────────────────────
# 0. Setup
# ────────────────────────────────────────────────────────────
import os, random, time, gc
import pandas as pd
from PIL import Image, UnidentifiedImageError
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import lightning as L
from torchmetrics.regression import MeanAbsoluteError, MeanSquaredError

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# ===>>> CONFIGURE THESE <<<===
POSTER_DIR = "/kaggle/input/computer-vision-project-dataset/poster_images/"
IMG_COL    = "image_path"
SCORE_COL  = "imdb_score"
EMBEDDING_DIM = 768
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 1e-4
# =============================

# ────────────────────────────────────────────────────────────
# 1. Dataset Class
# ────────────────────────────────────────────────────────────
class PosterDS(Dataset):
    def __init__(self, df, root, tfm, embedding_dim):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.tfm = tfm
        self.embedding_dim = embedding_dim

        min_score = self.df["actor_score"].min()
        max_score = self.df["actor_score"].max()
        self.df["actor_score_norm"] = (self.df["actor_score"] - min_score) / (max_score - min_score)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row[IMG_COL])

        try:
            img = Image.open(path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError):
            return self.__getitem__((idx + 1) % len(self))

        x_img = self.tfm(img)
        y = torch.tensor(row[SCORE_COL], dtype=torch.float32)

        emb = torch.tensor(row["title_embedding"], dtype=torch.float32)
        if emb.size(0) > self.embedding_dim:
            emb = emb[:self.embedding_dim]
        elif emb.size(0) < self.embedding_dim:
            pad = torch.zeros(self.embedding_dim - emb.size(0))
            emb = torch.cat([emb, pad])

        actor_score = torch.tensor(row["actor_score_norm"], dtype=torch.float32).unsqueeze(0)
        return x_img, emb, actor_score, y

# ────────────────────────────────────────────────────────────
# 2. ViT + Title + Face Fusion Model
# ────────────────────────────────────────────────────────────
class ViTWithTitleAndFace(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True)
        vit_out_dim = self.vit.head.in_features
        self.vit.head = nn.Identity()

        self.face_proj = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.head = nn.Sequential(
            nn.Linear(vit_out_dim + embedding_dim + 128, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def forward(self, x_img, x_emb, x_actor):
        x_vit = self.vit(x_img)
        x_face = self.face_proj(x_actor)
        x = torch.cat([x_vit, x_emb, x_face], dim=1)
        return self.head(x).squeeze(1)

# ────────────────────────────────────────────────────────────
# 3. LightningModule
# ────────────────────────────────────────────────────────────
class Regr(L.LightningModule):
    def __init__(self, model, lr=LR):
        super().__init__()
        self.model = model
        self.lr = lr
        self.mae = MeanAbsoluteError()
        self.mse = MeanSquaredError()

    def forward(self, x_img, x_emb, x_actor):
        return self.model(x_img, x_emb, x_actor)

    def _step(self, batch, tag):
        x_img, x_emb, x_actor, y = batch
        yhat = self(x_img, x_emb, x_actor)
        loss = nn.MSELoss()(yhat, y)
        self.log(f"{tag}_mae", self.mae(yhat, y), prog_bar=True, batch_size=len(y))
        self.log(f"{tag}_mse", self.mse(yhat, y), prog_bar=True, batch_size=len(y))
        return loss

    def training_step(self, b, i): return self._step(b, "train")
    def validation_step(self, b, i): return self._step(b, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

# ────────────────────────────────────────────────────────────
# 4. Dataloader Builder
# ────────────────────────────────────────────────────────────
def build_loaders(df, img_size=224, bs=32, workers=2, embedding_dim=768):
    df = df[df[IMG_COL].apply(lambda p: os.path.exists(os.path.join(POSTER_DIR, p)))]
    val_df = df.sample(frac=0.1, random_state=SEED)
    tr_df  = df.drop(val_df.index)

    mean, std = [0.5]*3, [0.5]*3
    tr_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(.2, .2, .2, .1),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), transforms.Normalize(mean, std)])

    tr_loader = DataLoader(
        PosterDS(tr_df, POSTER_DIR, tr_tfm, embedding_dim),
        batch_size=bs, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader = DataLoader(
        PosterDS(val_df, POSTER_DIR, val_tfm, embedding_dim),
        batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
    return tr_loader, val_loader

# ────────────────────────────────────────────────────────────
# 5. Training Runner
# ────────────────────────────────────────────────────────────
def train_one(df, embedding_dim=768, epochs=EPOCHS, bs=BATCH_SIZE, freeze_vit=False):
    tr_loader, val_loader = build_loaders(df, bs=bs, embedding_dim=embedding_dim)
    model = ViTWithTitleAndFace(embedding_dim)

    if freeze_vit:
        for param in model.vit.parameters():
            param.requires_grad = False
        print("🧊 ViT backbone is frozen.")
    else:
        print("🔥 ViT backbone is trainable.")

    lit_model = Regr(model)

    checkpoint_cb = L.pytorch.callbacks.ModelCheckpoint(
        monitor="val_mae", mode="min", save_top_k=1,
        filename="best-vit-title-face-{epoch:02d}-{val_mae:.3f}"
    )

    print(f"🧪 Training on {len(tr_loader.dataset):,} samples")
    print(f"🧾 Validating on {len(val_loader.dataset):,} samples")

    trainer = L.Trainer(
        max_epochs=epochs,
        precision="16-mixed" if torch.cuda.is_available() else 32,
        accelerator="auto",
        deterministic=True,
        log_every_n_steps=10,
        callbacks=[checkpoint_cb]
    )

    t0 = time.time()
    trainer.fit(lit_model, tr_loader, val_loader)
    metrics = trainer.validate(lit_model, val_loader, verbose=False)[0]
    print(f"\n▶ FINAL MAE={metrics['val_mae']:.3f}  MSE={metrics['val_mse']:.3f}  |  {(time.time() - t0)/60:.1f} min")
    print(f"📦 Best checkpoint saved to: {checkpoint_cb.best_model_path}")

    # Sample predictions
    lit_model.eval()
    with torch.no_grad():
        for x_img, x_emb, x_actor, y_true in val_loader:
            y_pred = lit_model(x_img.to(lit_model.device), x_emb.to(lit_model.device), x_actor.to(lit_model.device))
            print("\n🔍 Sample predictions:")
            for i in range(min(5, len(y_true))):
                print(f"🎯 True: {y_true[i].item():.2f} → 🧠 Pred: {y_pred[i].item():.2f}")
            break

    return metrics

# ────────────────────────────────────────────────────────────
# 6. Run Training
# ────────────────────────────────────────────────────────────
metrics = train_one(final_df)

print("\nFinal Metrics:")
print("MAE:", metrics["val_mae"], "MSE:", metrics["val_mse"])

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


🔥 ViT backbone is trainable.
🧪 Training on 44,380 samples
🧾 Validating on 4,931 samples


2025-06-10 09:07:39.120479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749546459.302548      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749546459.352917      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type                | Params | Mode 
------------------------------------------------------
0 | model | ViTWithTitleAndFace | 86.7 M | train
1 | mae   | MeanAbsoluteError   | 0      | train
2 | mse   | MeanSquaredError    | 0      | train
------------------------------------------------------
86.7 M    Trainable params
0         Non-trainable params
86.7 M    Total pa

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]


▶ FINAL MAE=1.786  MSE=6.218  |  129.5 min
📦 Best checkpoint saved to: /kaggle/working/lightning_logs/version_0/checkpoints/best-vit-title-face-epoch=04-val_mae=1.711.ckpt

🔍 Sample predictions:
🎯 True: 2.50 → 🧠 Pred: 1.71
🎯 True: 5.60 → 🧠 Pred: 6.05
🎯 True: 6.60 → 🧠 Pred: 6.39
🎯 True: 7.10 → 🧠 Pred: 5.72
🎯 True: 0.00 → 🧠 Pred: 5.52

Final Metrics:
MAE: 1.785974144935608 MSE: 6.218483924865723
